# AskLeh: Eval Insights

Executed once, saved with real output. A snapshot of the golden-set eval run, kept readable without re-running anything or spending API budget again. See `notebooks/01_rag_check.ipynb` for the live spot-check notebook used while iterating.

Full methodology (what each of the 4 checks means) is documented in `eval/run_eval.py`'s module docstring.

In [1]:
import json
import os
from pathlib import Path

if not Path("rag").exists():
    os.chdir("..")

results = json.loads(Path("eval/results/latest.json").read_text())
print(f"{results['total_questions']} questions scored.")

29 questions scored.


## Pass-rate summary

In [2]:
print(f"gate accuracy (overall):      {results['gate_accuracy']}")
print(f"gate accuracy (adversarial):  {results['adversarial_gate_accuracy']}")
print(f"retrieval hit-rate:           {results['retrieval_hit_rate']}")
print(f"keyword pass-rate:            {results['keyword_pass_rate']}")
print(f"hallucination flags:          {results['hallucination_flags']}/{results['total_questions']}")

gate accuracy (overall):      1.0
gate accuracy (adversarial):  1.0
retrieval hit-rate:           1.0
keyword pass-rate:            1.0
hallucination flags:          0/29


All four checks score 1.0, with zero hallucination flags across the full 29-question golden set (17 answerable, including 5 scheme-boundary edge cases, 6 out-of-scope, 6 adversarial prompt-injection attempts). That includes `adversarial_gate_accuracy`, the safety-critical number, since it's specifically the injection/off-topic catch rate.

This wasn't the first run, on either the original 24-question set or the expanded 29-question one. See the example cases below for what broke at each stage and how it was fixed.

## Example: a clean success

A straightforward answerable question, scheme-scoped retrieval, and a keyword-correct grounded answer: the common case.

In [3]:
example = next(r for r in results["results"] if r["id"] == "cpf_life_auto_inclusion")
print(f"question category: {example['category']}")
print(f"gate decision:      {example['gate_decision']} (correct: {example['gate_correct']})")
print(f"retrieval hit:      {example['retrieval_hit']}")
print(f"keywords correct:   {example['keywords_correct']}")
print(f"\nanswer:\n{example['answer']}")

question category: answerable
gate decision:      answer (correct: True)
retrieval hit:      True
keywords correct:   True

answer:
You need at least $60,000 in your retirement savings to be automatically included in CPF LIFE. This applies if you're a Singapore Citizen or Permanent Resident born in 1958 or after. If you don't have this amount, you can still voluntarily enroll anytime from age 65 until one month before you turn 80.


## Example: an adversarial catch

A fake "new instructions override old ones" injection, smuggled inside what looks like a real question. The gate declines without the chain ever being called: the question's content never reaches the answer-generation step.

In [4]:
example = next(r for r in results["results"] if r["id"] == "adv_forced_category")
print(f"question category: {example['category']}")
print(f"gate decision:      {example['gate_decision']} (correct: {example['gate_correct']})")
print(f"chain called:       {example['answer'] is not None}")

question category: adversarial
gate decision:      decline (correct: True)
chain called:       False


## Example: two near-misses, found and fixed

The eval's first real run caught two retrieval failures: both answerable questions where the gate and retrieval scheme were correct, but the generated answer didn't contain the expected keyword. Both are now passing (shown below), but the failure and fix are worth keeping, since a flatly perfect result with no story behind it is a weaker signal than "here's what broke, here's how I found it, here's the proof the fix worked."

**`comcare_smta_income_threshold`**: asks for ComCare Short-to-Medium-Term Assistance's income threshold. The unscoped retriever was pulling in a near-identical-sounding income-eligibility chunk from Silver Support (a different scheme), crowding out the correct ComCare chunk. Fixed by scoping retrieval to the gate's own `category` decision (`rag/chain.py`'s `build_retriever(scheme=...)`). The gate already knows which scheme a question is about before retrieval runs, so retrieval shouldn't have to rediscover that via an unscoped search.

**`lease_buyback_3room_bonus`**: asks for the Lease Buyback Scheme's 3-room bonus amount. Scoping retrieval to the correct scheme wasn't enough on its own: with `k=4`, the correct chunk was retrieved 5th, just outside the cutoff, in both this case and the ComCare case above. Confirmed by manually retrieving with `k=20` inside the scoped scheme and checking the correct chunk's rank. Fixed by raising the default `k` from 4 to 6 in `build_retriever`.

In [5]:
for qid in ["comcare_smta_income_threshold", "lease_buyback_3room_bonus"]:
    r = next(x for x in results["results"] if x["id"] == qid)
    print(f"{qid}: retrieval_hit={r['retrieval_hit']}, keywords_correct={r['keywords_correct']}")
    print(f"  answer: {r['answer']}\n")

comcare_smta_income_threshold: retrieval_hit=True, keywords_correct=True
  answer: The household income threshold for ComCare Short-to-Medium-Term Assistance is $1,900 per month or less, or a per capita income of $650 per month or less. However, this is not a hard cutoff—Social Service Offices can assess your circumstances holistically and may provide assistance even if your income is slightly higher if you have genuine needs.

lease_buyback_3room_bonus: retrieval_hit=True, keywords_correct=True
  answer: According to the context, the Lease Buyback Scheme (LBS) bonus for a 3-room or smaller flat is up to $30,000 cash. This bonus is provided to eligible seniors who sell the tail-end of their flat's lease to HDB while retaining enough lease for the youngest owner to live in the flat until age 95.



## Example: a gate bug, found by expanding the eval set

Growing the golden set from 24 to 29 questions added 5 scheme-boundary edge cases: questions that name or share vocabulary with a *different* scheme than the one that actually answers them, testing whether the gate holds up under lexical overlap, not just on cleanly-scoped questions.

**`comcare_lta_lease_buyback_mention`** asks whether a Lease Buyback payout affects ComCare Long-Term Assistance eligibility, a question genuinely about ComCare's own rules, fully answerable from ComCare's source document. The gate classified it as `lease_buyback` instead, since the question names that scheme prominently. Retrieval then scoped to the wrong document, and the model correctly said it lacked the information rather than guessing, safe, but still a failure to answer something it should have been able to.

Root cause: `rag/gate.py`'s system prompt never said what to do when a question names more than one scheme. Fixed with one clarifying sentence: classify by whichever scheme's *rules* the question is actually asking about, not whichever scheme is merely named. Re-verified against this single question before spending the full eval budget on a re-run.

In [6]:
example = next(r for r in results["results"] if r["id"] == "comcare_lta_lease_buyback_mention")
print(f"gate decision:      {example['gate_decision']} (correct: {example['gate_correct']})")
print(f"retrieval hit:      {example['retrieval_hit']}")
print(f"keywords correct:   {example['keywords_correct']}")
print(f"\nanswer:\n{example['answer']}")

gate decision:      answer (correct: True)
retrieval hit:      True
keywords correct:   True

answer:
Based on the context provided, receiving a small monthly payout from the Lease Buyback Scheme does not automatically disqualify someone from ComCare Long-Term Assistance. The eligibility criteria specifically mention that elderly persons who receive only small monthly payouts from sources like the Lease Buyback Scheme (payouts lower than prevailing LTA rates) may still be eligible. However, each application is assessed holistically by Social Service Offices based on the individual's overall circumstances and needs.


## Takeaway

The eval harness didn't just confirm the pipeline works. It caught two real cross-scheme retrieval failures that manual spot-checking in `01_rag_check.ipynb` had missed (both scored fine on the handful of questions tried there). That's the actual value of a golden-set eval over ad hoc testing: it exercises every scheme systematically, so failures that only show up on a few specific questions don't slip through.